# EXP-33: Barnase–Barstar Double Mutant Cycle (DMC)
**10 single + 5 double mutations via alchemical FEP**

- **Barnase mutations:** K27A, R59A, H102A, R83A, R87A
- **Barstar mutations:** D35A, D39A, Y29A, E76A, T42A
- **5 double mutations:** (K27A+D35A), (R59A+D39A), (H102A+Y29A), (R83A+E76A), (R87A+T42A)
- **PASS:** R > 0.7, RMSE < 2.0 kcal/mol, top 5 coupled pairs correct
- **MARGINAL:** R > 0.5, RMSE < 3.0 | **FAIL:** otherwise
- **PDB: 1BRS chains A (barnase) + D (barstar)**

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging, time
from pathlib import Path
import numpy as np

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-33'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['prep','fep_complex','fep_free_barnase','fep_free_barstar','analysis','figures']:
    (WORK_DIR/d).mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO)

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')

def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

In [ ]:
import openmm
from openmm import unit, Platform
Platform.getPlatformByName('CUDA')
print(f'OpenMM {openmm.__version__}, CUDA ready')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
import mdtraj as md
from src.config import (SystemConfig, MinimizationConfig, EquilibrationConfig, FEPConfig, KCAL_TO_KJ)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.fep import run_fep_campaign
from src.simulate.platform import select_platform
from src.analyze.fep import compute_delta_g_mbar, compute_delta_delta_g
from openmm.app import PME, ForceField, Simulation, HBonds, PDBFile
from openmm import LangevinMiddleIntegrator, XmlSerializer
print('Imports OK')

In [ ]:
TEMPERATURE_K = 300.0  # Barnase-barstar experiments at 300K
system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(nvt_duration_ps=500.0, npt_duration_ps=1000.0, temperature_k=TEMPERATURE_K)
fep_config = FEPConfig(n_lambda_windows=16, per_window_duration_ns=2.0, temperature_k=TEMPERATURE_K)  # optimized from 20: sufficient MBAR overlap for X→Ala

BACKBONE_NAMES = {'N', 'CA', 'C', 'O', 'H', 'HA', 'HA2', 'HA3', 'OXT', 'CB'}

# Mutation definitions: chain, resid, experimental DDG (kcal/mol)
SINGLE_MUTATIONS = {
    'K27A':  {'chain': 'barnase', 'resid': 27,  'ddg_exp': 3.8},
    'R59A':  {'chain': 'barnase', 'resid': 59,  'ddg_exp': 4.2},
    'H102A': {'chain': 'barnase', 'resid': 102, 'ddg_exp': 4.5},
    'R83A':  {'chain': 'barnase', 'resid': 83,  'ddg_exp': 5.0},
    'R87A':  {'chain': 'barnase', 'resid': 87,  'ddg_exp': 4.0},
    'D35A':  {'chain': 'barstar', 'resid': 35,  'ddg_exp': 5.5},
    'D39A':  {'chain': 'barstar', 'resid': 39,  'ddg_exp': 6.3},
    'Y29A':  {'chain': 'barstar', 'resid': 29,  'ddg_exp': 2.8},
    'E76A':  {'chain': 'barstar', 'resid': 76,  'ddg_exp': 2.0},
    'T42A':  {'chain': 'barstar', 'resid': 42,  'ddg_exp': 1.5},
}

DOUBLE_MUTATIONS = {
    'K27A+D35A':  {'muts': ['K27A', 'D35A'], 'ddg_exp': 12.0},
    'R59A+D39A':  {'muts': ['R59A', 'D39A'], 'ddg_exp': 13.5},
    'H102A+Y29A': {'muts': ['H102A', 'Y29A'], 'ddg_exp': 9.0},
    'R83A+E76A':  {'muts': ['R83A', 'E76A'], 'ddg_exp': 10.0},
    'R87A+T42A':  {'muts': ['R87A', 'T42A'], 'ddg_exp': 7.5},
}

print(f'{len(SINGLE_MUTATIONS)} single + {len(DOUBLE_MUTATIONS)} double mutations')

In [ ]:
# Build barnase-barstar complex from 1BRS (chains A + D)
PREP_DIR = WORK_DIR / 'prep'
EXP13_DIR = Path('/content/drive/MyDrive/v3_gpu_results/EXP-13')
exp13_sys = EXP13_DIR / 'system.xml'
exp13_state_candidates = [EXP13_DIR / 'eq_state.xml'] + list(EXP13_DIR.rglob('npt_final_state.xml'))
exp13_state = next((p for p in exp13_state_candidates if p.exists()), None)
topo_candidates = list(EXP13_DIR.rglob('topology_reference.pdb')) + list(EXP13_DIR.rglob('*solvated*.pdb'))

if exp13_sys.exists() and exp13_state:
    complex_system = XmlSerializer.deserialize(exp13_sys.read_text())
    cx_state = XmlSerializer.deserialize(exp13_state.read_text())
    complex_positions = cx_state.getPositions(asNumpy=True)
    complex_topo_pdb = str(topo_candidates[0]) if topo_candidates else None
    print('Complex loaded from EXP-13')
else:
    raw_pdb = fetch_pdb('1BRS', output_dir=PREP_DIR)
    cleaned = clean_structure(raw_pdb, chains_to_keep=['A', 'D'], remove_heteroatoms=True, remove_waters=True)
    _, complex_system, modeller = build_topology(cleaned, system_config)
    modeller, nw, _, _ = solvate_system(modeller, system_config)
    ff = ForceField(system_config.force_field, system_config.water_model)
    complex_system = ff.createSystem(modeller.topology, nonbondedMethod=PME,
        nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
    intg = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
    sim = Simulation(modeller.topology, complex_system, intg, select_platform('CUDA'))
    sim.context.setPositions(modeller.positions)
    complex_topo_pdb = str(PREP_DIR / 'bb_complex.pdb')
    with open(complex_topo_pdb, 'w') as f: PDBFile.writeFile(modeller.topology, modeller.positions, f)
    minimize_energy(sim, min_config)
    run_nvt(sim, eq_config, WORK_DIR/'fep_complex'/'nvt')
    run_npt(sim, eq_config, WORK_DIR/'fep_complex'/'npt')
    complex_positions = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
    # Save system XML
    with open(WORK_DIR/'fep_complex'/'system.xml', 'w') as f:
        f.write(XmlSerializer.serialize(complex_system))
    print(f'Complex built ({nw} waters)')

In [ ]:
# Free barnase (chain A) in water
raw_pdb_bn = fetch_pdb('1BRS', output_dir=PREP_DIR)
cleaned_bn = clean_structure(raw_pdb_bn, chains_to_keep=['A'], remove_heteroatoms=True, remove_waters=True)
_, free_bn_sys, mod_bn = build_topology(cleaned_bn, system_config)
mod_bn, nw_bn, _, _ = solvate_system(mod_bn, system_config)
ff = ForceField(system_config.force_field, system_config.water_model)
free_bn_sys = ff.createSystem(mod_bn.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
intg_bn = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
sim_bn = Simulation(mod_bn.topology, free_bn_sys, intg_bn, select_platform('CUDA'))
sim_bn.context.setPositions(mod_bn.positions)
free_bn_pdb = str(PREP_DIR / 'barnase_free.pdb')
with open(free_bn_pdb, 'w') as f: PDBFile.writeFile(mod_bn.topology, mod_bn.positions, f)
minimize_energy(sim_bn, min_config)
run_nvt(sim_bn, eq_config, WORK_DIR/'fep_free_barnase'/'nvt')
run_npt(sim_bn, eq_config, WORK_DIR/'fep_free_barnase'/'npt')
free_bn_positions = sim_bn.context.getState(getPositions=True).getPositions(asNumpy=True)
print(f'Free barnase equilibrated')

In [ ]:
# Free barstar (chain D) in water
raw_pdb_bs = fetch_pdb('1BRS', output_dir=PREP_DIR)
cleaned_bs = clean_structure(raw_pdb_bs, chains_to_keep=['D'], remove_heteroatoms=True, remove_waters=True)
_, free_bs_sys, mod_bs = build_topology(cleaned_bs, system_config)
mod_bs, nw_bs, _, _ = solvate_system(mod_bs, system_config)
ff = ForceField(system_config.force_field, system_config.water_model)
free_bs_sys = ff.createSystem(mod_bs.topology, nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True)
intg_bs = LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
sim_bs = Simulation(mod_bs.topology, free_bs_sys, intg_bs, select_platform('CUDA'))
sim_bs.context.setPositions(mod_bs.positions)
free_bs_pdb = str(PREP_DIR / 'barstar_free.pdb')
with open(free_bs_pdb, 'w') as f: PDBFile.writeFile(mod_bs.topology, mod_bs.positions, f)
minimize_energy(sim_bs, min_config)
run_nvt(sim_bs, eq_config, WORK_DIR/'fep_free_barstar'/'nvt')
run_npt(sim_bs, eq_config, WORK_DIR/'fep_free_barstar'/'npt')
free_bs_positions = sim_bs.context.getState(getPositions=True).getPositions(asNumpy=True)
print(f'Free barstar equilibrated')

In [ ]:
# Load topologies for atom selection
cx_top = md.load(complex_topo_pdb).topology
bn_top = md.load(free_bn_pdb).topology
bs_top = md.load(free_bs_pdb).topology

# Identify chain indices in complex
cx_chains = list(cx_top.chains)
# Chain A (barnase) is first, chain D (barstar) is second
barnase_chain_cx = 0
barstar_chain_cx = 1

def get_sidechain(topology, resSeq, chain_id):
    """Atoms beyond backbone + CB."""
    indices = []
    for atom in topology.atoms:
        if atom.residue.resSeq == resSeq and atom.residue.chain.index == chain_id:
            if atom.name not in BACKBONE_NAMES:
                indices.append(atom.index)
    if not indices:
        # Fallback: all atoms of that residue
        for atom in topology.atoms:
            if atom.residue.resSeq == resSeq and atom.residue.chain.index == chain_id:
                indices.append(atom.index)
    return indices

print(f'Barnase chain: {barnase_chain_cx}, Barstar chain: {barstar_chain_cx}')

In [ ]:
# Run single mutations
ddg_singles = {}
checkpoint = DRIVE_OUTPUT / 'checkpoint_singles.json'
if checkpoint.exists():
    with open(checkpoint) as f: ddg_singles = json.load(f)
    print(f'Resumed {len(ddg_singles)} single mutations')

for mut_name, info in SINGLE_MUTATIONS.items():
    if mut_name in ddg_singles:
        print(f'{mut_name}: already computed, skipping')
        continue

    t0 = time.time()
    resid = info['resid']
    is_barnase = info['chain'] == 'barnase'
    cx_chain = barnase_chain_cx if is_barnase else barstar_chain_cx
    print(f'\n=== {mut_name} (resid {resid}, {info["chain"]}) ===')

    # Complex leg
    sc_cx = get_sidechain(cx_top, resid, cx_chain)
    fep_cx = run_fep_campaign(system=complex_system, positions=complex_positions,
        mutant_atom_indices=sc_cx, config=fep_config,
        output_dir=WORK_DIR/'fep_complex'/mut_name)
    dg_cx = compute_delta_g_mbar(fep_cx['energy_matrix'], fep_cx['n_samples_per_state'], TEMPERATURE_K)

    # Free leg
    if is_barnase:
        sc_fr = get_sidechain(bn_top, resid, 0)
        fep_fr = run_fep_campaign(system=free_bn_sys, positions=free_bn_positions,
            mutant_atom_indices=sc_fr, config=fep_config,
            output_dir=WORK_DIR/'fep_free_barnase'/mut_name)
    else:
        sc_fr = get_sidechain(bs_top, resid, 0)
        fep_fr = run_fep_campaign(system=free_bs_sys, positions=free_bs_positions,
            mutant_atom_indices=sc_fr, config=fep_config,
            output_dir=WORK_DIR/'fep_free_barstar'/mut_name)
    dg_fr = compute_delta_g_mbar(fep_fr['energy_matrix'], fep_fr['n_samples_per_state'], TEMPERATURE_K)

    ddg = compute_delta_delta_g(dg_cx, dg_fr)
    elapsed = time.time() - t0
    print(f'  \u0394\u0394G: {ddg["delta_delta_g_kcal_mol"]:.2f} \u00b1 {ddg["delta_delta_g_std_kcal_mol"]:.2f} ({elapsed:.0f}s)')

    ddg_singles[mut_name] = {
        'ddg_sim': ddg['delta_delta_g_kcal_mol'], 'ddg_std': ddg['delta_delta_g_std_kcal_mol'],
        'dg_complex': dg_cx['delta_g_kcal_mol'], 'dg_free': dg_fr['delta_g_kcal_mol'],
        'ddg_exp': info['ddg_exp'],
    }
    with open(checkpoint, 'w') as f: json.dump(ddg_singles, f, indent=2)

print(f'\nCompleted {len(ddg_singles)} single mutations')

In [ ]:
# Run double mutations
ddg_doubles = {}
checkpoint_d = DRIVE_OUTPUT / 'checkpoint_doubles.json'
if checkpoint_d.exists():
    with open(checkpoint_d) as f: ddg_doubles = json.load(f)
    print(f'Resumed {len(ddg_doubles)} double mutations')

for dbl_name, dbl_info in DOUBLE_MUTATIONS.items():
    if dbl_name in ddg_doubles:
        print(f'{dbl_name}: already computed, skipping')
        continue

    t0 = time.time()
    mut1, mut2 = dbl_info['muts']
    info1 = SINGLE_MUTATIONS[mut1]
    info2 = SINGLE_MUTATIONS[mut2]
    print(f'\n=== {dbl_name} ===')

    # Complex leg: combine both residue sidechains
    cx_chain1 = barnase_chain_cx if info1['chain'] == 'barnase' else barstar_chain_cx
    cx_chain2 = barnase_chain_cx if info2['chain'] == 'barnase' else barstar_chain_cx
    sc_cx = get_sidechain(cx_top, info1['resid'], cx_chain1) + get_sidechain(cx_top, info2['resid'], cx_chain2)
    fep_cx = run_fep_campaign(system=complex_system, positions=complex_positions,
        mutant_atom_indices=sc_cx, config=fep_config,
        output_dir=WORK_DIR/'fep_complex'/dbl_name)
    dg_cx = compute_delta_g_mbar(fep_cx['energy_matrix'], fep_cx['n_samples_per_state'], TEMPERATURE_K)

    # Free legs: each mutation in its own free protein
    # For double mutations, free leg = sum of individual free legs
    dg_free_total_kj = 0.0
    dg_free_std_sq = 0.0
    for mut_n, mut_i in [(mut1, info1), (mut2, info2)]:
        if mut_i['chain'] == 'barnase':
            sc_f = get_sidechain(bn_top, mut_i['resid'], 0)
            fep_f = run_fep_campaign(system=free_bn_sys, positions=free_bn_positions,
                mutant_atom_indices=sc_f, config=fep_config,
                output_dir=WORK_DIR/'fep_free_barnase'/f'{dbl_name}_{mut_n}')
        else:
            sc_f = get_sidechain(bs_top, mut_i['resid'], 0)
            fep_f = run_fep_campaign(system=free_bs_sys, positions=free_bs_positions,
                mutant_atom_indices=sc_f, config=fep_config,
                output_dir=WORK_DIR/'fep_free_barstar'/f'{dbl_name}_{mut_n}')
        dg_f = compute_delta_g_mbar(fep_f['energy_matrix'], fep_f['n_samples_per_state'], TEMPERATURE_K)
        dg_free_total_kj += dg_f['delta_g_kj_mol']
        dg_free_std_sq += dg_f['delta_g_std_kj_mol']**2

    # Construct combined free result for DDG
    dg_free_combined = {
        'delta_g_kj_mol': dg_free_total_kj,
        'delta_g_std_kj_mol': np.sqrt(dg_free_std_sq),
    }
    ddg = compute_delta_delta_g(dg_cx, dg_free_combined)
    elapsed = time.time() - t0
    print(f'  \u0394\u0394G: {ddg["delta_delta_g_kcal_mol"]:.2f} \u00b1 {ddg["delta_delta_g_std_kcal_mol"]:.2f} ({elapsed:.0f}s)')

    ddg_doubles[dbl_name] = {
        'ddg_sim': ddg['delta_delta_g_kcal_mol'], 'ddg_std': ddg['delta_delta_g_std_kcal_mol'],
        'dg_complex': dg_cx['delta_g_kcal_mol'], 'ddg_exp': dbl_info['ddg_exp'],
    }
    with open(checkpoint_d, 'w') as f: json.dump(ddg_doubles, f, indent=2)

print(f'\nCompleted {len(ddg_doubles)} double mutations')

In [ ]:
# Coupling energy analysis
# \u0394\u0394G_coupling = \u0394\u0394G_double - \u0394\u0394G_single1 - \u0394\u0394G_single2
coupling_results = {}
for dbl_name, dbl_info in DOUBLE_MUTATIONS.items():
    mut1, mut2 = dbl_info['muts']
    if dbl_name in ddg_doubles and mut1 in ddg_singles and mut2 in ddg_singles:
        coupling = (ddg_doubles[dbl_name]['ddg_sim'] -
                    ddg_singles[mut1]['ddg_sim'] - ddg_singles[mut2]['ddg_sim'])
        coupling_exp = (dbl_info['ddg_exp'] -
                        SINGLE_MUTATIONS[mut1]['ddg_exp'] - SINGLE_MUTATIONS[mut2]['ddg_exp'])
        coupling_results[dbl_name] = {
            'coupling_sim': coupling, 'coupling_exp': coupling_exp,
        }
        print(f'{dbl_name}: coupling = {coupling:.2f} (exp: {coupling_exp:.2f}) kcal/mol')

In [ ]:
# Overall correlation (singles + doubles combined)
all_names = list(ddg_singles.keys()) + list(ddg_doubles.keys())
sim_all = np.array([ddg_singles.get(n, ddg_doubles.get(n, {})).get('ddg_sim', 0) for n in all_names])
exp_all = np.array([ddg_singles.get(n, ddg_doubles.get(n, {})).get('ddg_exp', 0) for n in all_names])

r_val, p_val = sp_stats.pearsonr(exp_all, sim_all)
rmse = np.sqrt(np.mean((sim_all - exp_all)**2))

# Top 5 coupled: rank by |coupling_sim|, check if top 5 match
if coupling_results:
    coupling_sim_rank = sorted(coupling_results.keys(), key=lambda x: abs(coupling_results[x]['coupling_sim']), reverse=True)
    coupling_exp_rank = sorted(coupling_results.keys(), key=lambda x: abs(coupling_results[x]['coupling_exp']), reverse=True)
    top5_correct = sum(1 for c in coupling_sim_rank[:5] if c in coupling_exp_rank[:5])
else:
    top5_correct = 0

print(f'Pearson R: {r_val:.3f}')
print(f'RMSE: {rmse:.2f} kcal/mol')
print(f'Top 5 coupled pairs correct: {top5_correct}/5')

if r_val > 0.7 and rmse < 2.0 and top5_correct >= 5:
    verdict = 'PASS'
elif r_val > 0.5 and rmse < 3.0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'\nVERDICT: {verdict}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DDG correlation
ax = axes[0]
s_names = list(ddg_singles.keys()); d_names = list(ddg_doubles.keys())
s_sim = [ddg_singles[n]['ddg_sim'] for n in s_names]
s_exp = [ddg_singles[n]['ddg_exp'] for n in s_names]
d_sim = [ddg_doubles[n]['ddg_sim'] for n in d_names]
d_exp = [ddg_doubles[n]['ddg_exp'] for n in d_names]
ax.scatter(s_exp, s_sim, c='steelblue', s=60, label='Singles', zorder=5)
ax.scatter(d_exp, d_sim, c='coral', s=80, marker='D', label='Doubles', zorder=5)
for i, n in enumerate(s_names): ax.annotate(n, (s_exp[i], s_sim[i]), fontsize=6, xytext=(4,3), textcoords='offset points')
for i, n in enumerate(d_names): ax.annotate(n, (d_exp[i], d_sim[i]), fontsize=6, xytext=(4,3), textcoords='offset points')
lims = [min(min(exp_all)-1, min(sim_all)-1), max(max(exp_all)+1, max(sim_all)+1)]
ax.plot(lims, lims, 'k--', alpha=0.3)
ax.set_xlabel('Exp \u0394\u0394G (kcal/mol)'); ax.set_ylabel('Sim \u0394\u0394G (kcal/mol)')
ax.set_title(f'R={r_val:.2f}, RMSE={rmse:.1f}'); ax.legend()

# Coupling energies
ax = axes[1]
if coupling_results:
    c_names = list(coupling_results.keys())
    c_sim = [coupling_results[n]['coupling_sim'] for n in c_names]
    c_exp = [coupling_results[n]['coupling_exp'] for n in c_names]
    ax.bar(np.arange(len(c_names))-0.2, c_exp, 0.35, label='Exp', color='steelblue')
    ax.bar(np.arange(len(c_names))+0.2, c_sim, 0.35, label='Sim', color='coral')
    ax.set_xticks(range(len(c_names))); ax.set_xticklabels(c_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Coupling Energy (kcal/mol)'); ax.legend()
    ax.set_title('Double Mutant Coupling Energies')

fig.suptitle(f'EXP-33: Barnase\u2013Barstar DMC \u2014 {verdict}', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
sync_stop.set()  # Stop periodic sync before final copy

results = {
    'experiment_id': EXP_ID, 'pearson_r': round(float(r_val), 4),
    'rmse_kcal': round(float(rmse), 3), 'top5_coupled_correct': top5_correct,
    'n_singles': len(ddg_singles), 'n_doubles': len(ddg_doubles),
    'verdict': verdict,
    'singles': ddg_singles, 'doubles': ddg_doubles, 'coupling': coupling_results,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

ckpt.mark_done('complete')

# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))